# TSST - Práctica 8: Transformers y su aplicación a detección de actividad de voz (VAD)

**Alicia Lozano Díez**

## Objetivo

Los objetivos de esta práctica son:
1. Comprender la Arquitectura **Transformer**
2. Extraer **Embeddings** usando un modelo pre-entrenado basado en Transformers
3. Usar los embeddings para la detección de actividad (VAD) mediante **Clustering**

### Materiales

- Guión (.ipynb) de la práctica - Moodle
- Audios de ejemplo de la práctica 7 - Moodle

In [2]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memoria disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No se detectó GPU.')

Dispositivo: cpu
No se detectó GPU.


## PARTE 1: La Arquitectura Transformer

La arquitectura **Transformer** fue propuesta en ["Attention Is All You Need" (Vaswani et al., 2017)](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf).

Esta arquitectura se basa en la idea de modelar una secuencia utilizando únicamente mecanismos de atención, analizando toda la secuencia al mismo tiempo (en contraste con las RNNs). Esta innovación permite:
- Paralelización completa y escalabilidad (acelerando el entrenamiento).
- Mitigar el problema de desvanecimiento de gradiente.
- Capturar mejor las dependencias a largo alcance.

___
**Arquitectura**:

<img src="https://raw.githubusercontent.com/harvardnlp/annotated-transformer/v4/images/ModalNet-21-big.png" width="500"/>


1. **Encoder**: Transforma la secuencia de entrada en representaciones contextualizadas.
2. **Decoder**: Genera la secuencia de salida utilizando la información producida por el Encoder.


___
Los componentes clave son:
- **Positional Encoding**: Añade información sobre la posición de cada token dentro de la secuencia.
- Mecanismo de atención (**Multi-head self-attention**): permite al modelo atender simultáneamente a distintas posiciones de la secuencia de entrada. Cada cabeza aprende distintos tipos de dependencias dentro de los datos.
- **Position-wise Feed-Forward Networks**: Introducen no linealidad y aumentan la capacidad expresiva del modelo.
- **Cross-Attention**: Permite que el decoder atienda a la salida del encoder, integrando la información de la secuencia de entrada durante la generación.


### 1.1 Mecanismo de Atención

**Scaled Dot-Product Attention**

<div align="center">
  <img src="https://raw.githubusercontent.com/harvardnlp/annotated-transformer/v4/images/ModalNet-19-big.png" width="150"/>
</div>

La atención simple calcula una suma ponderada de los **Values** (V) en función de la similitud entre **Queries** (Q) y **Keys** (K):

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [3]:
import math
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None, dropout=None):
    d_k = Q.size(-1)

    # TO DO: implementar la fórmula de atención simple
    # Cálculo de los scores de atención
    scores = ...

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Obtención de los pesos de atención
    attention_weights = ...

    if dropout is not None:
        attention_weights = dropout(attention_weights)

    # Cálculo del vector de salida de la atención
    output = ...
    return output, attention_weights

**Multi-Head Attention**

<div align="center">
  <img src="https://raw.githubusercontent.com/harvardnlp/annotated-transformer/v4/images/ModalNet-20-big.png" width="250"/>
</div>

Puesto que una única atención puede resultar limitada, el modelo ejecuta varias en paralelo (múltiples cabezas). Cada cabeza aprenderá proyecciones diferentes:

$$
\text{MultiHead}(Q, K, V)
=
\text{Concat}(\text{head}_1, \dots, \text{head}_h) W^{O}
$$

Donde,
$$\text{head}_i = \text{Attention}(QW_i^{(Q)}, KW_i^{(K)}, VW_i^{(V)})$$


In [4]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # TO DO: Definir las proyecciones lineales (Q, K, V y la de salida)
        self.W_q = ...
        self.W_k = ...
        self.W_v = ...
        self.W_o = ...

        self.dropout = nn.Dropout(p=dropout)

    def split_heads(self, x):
      # (batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_k)
        batch_size, seq_len, _ = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # TO DO: Calcular Q, K, V
        Q = ...
        K = ...
        V = ...

        # Aplicar la función de atención ya definida
        x, attn_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout=self.dropout
        )

        # Concatenación de cabezas
        x = x.transpose(1, 2).contiguous()
        x = x.view(batch_size, -1, self.d_model)

        # Proyección final de salida
        return self.W_o(x), attn_weights

Con el siguiente código podemos comprobar que las dimensiones son correctas:

In [5]:
batch_size = 2
seq_len = 4
d_model = 8
num_heads = 2

# Crear tensores aleatorios
x = torch.randn(batch_size, seq_len, d_model)

# Instanciar el módulo
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

# TO DO: Forward (self-attention)
output, attn_weights = ...

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", attn_weights.shape)

# Verificar que las probabilidades suman 1 (softmax)
print(attn_weights[0, 0])  # cabeza 0 del batch 0
print(attn_weights[0, 0].sum(dim=-1))

TypeError: cannot unpack non-iterable ellipsis object

**Comparación con nn.MultiheadAttention**: Para verificar que nuestra implementación es correcta, podemos compararla con la implementación oficial de PyTorch.


In [ ]:
official_mha = nn.MultiheadAttention(
    embed_dim=d_model,
    num_heads=num_heads,
    dropout=0.0,
    batch_first=True
)

# La implementación oficial usa una única matriz concatenada para Q, K y V, por lo que debemos copiar manualmente los pesos:
with torch.no_grad():
    official_mha.in_proj_weight.copy_(
        torch.cat([mha.W_q.weight,
                   mha.W_k.weight,
                   mha.W_v.weight], dim=0)
    )

    official_mha.in_proj_bias.copy_(
        torch.cat([mha.W_q.bias,
                   mha.W_k.bias,
                   mha.W_v.bias], dim=0)
    )

    official_mha.out_proj.weight.copy_(mha.W_o.weight)
    official_mha.out_proj.bias.copy_(mha.W_o.bias)


# Comparar salidas
mha.eval()
official_mha.eval()

out_custom, attn_custom = mha(x, x, x)
out_official, attn_official = official_mha(x, x, x)

print(torch.allclose(out_custom, out_official, atol=1e-6))

En modelos generativos (como el decoder del Transformer), el objetivo es predecir cada token utilizando únicamente la información disponible hasta esa posición de la secuencia. Es decir, la predicción en el instante *t* debe depender sólo de los tokens anteriores *(1, ..., t)*

Sin la aplicación de una máscara, el mecanismo de self-attention permitiría que cada posición atendiera a **todos los tokens de la secuencia**, incluidos aquellos situados en posiciones futuras, generando una fuga de información durante el entrenamiento.

A continuación, se muestra un ejemplo de aplicación de enmascarado:


In [ ]:
# Crear máscara causal
mask = torch.tril(torch.ones(seq_len, seq_len))
mask = mask.unsqueeze(0).unsqueeze(0)

output, attn_weights = mha(x, x, x, mask=mask)

print("Attention weights con máscara:")
print(attn_weights[0, 0])

**PREGUNTAS:**
1. ¿Qué token atiende más a cuál? ¿Es la matriz de atención simétrica?
2. ¿Cómo influyen las proyecciones lineales $W_Q$, $W_K$ y $W_V$?

### 1.2 Encoder Layer


Utilizando los módulos anteriores, podemos crear la estructura de la capa de codificación (encoder):

In [ ]:

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # TO DO: Aplicar atención
        attn_out, _ = ...

        # TO DO: Aplicar conexión residual + normalización
        x = ...

        # TO DO: Aplicar feed_forward
        ff_out = ...

        # TO DO: Aplicar segunda conexión residual + normalización
        x = ...
        return x

De nuevo, podemos comprobar que las dimensiones son correctas:

In [ ]:
batch_size = 2
seq_len = 5
d_model = 16
num_heads = 4
d_ff = 64

x = torch.randn(batch_size, seq_len, d_model)

encoder = TransformerEncoderLayer(d_model, num_heads, d_ff)

out = encoder(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

**PREGUNTAS:**

1. ¿Dónde se encuentran las conexiones residuales?
2. ¿Por qué este bloque mantiene la dimensión del tensor?


In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

### 1.4 Positional Encoding

Dado que el modelo no contiene recurrencia ni convolución, debemos introducir cierta información sobre la posición de los tokens en la secuencia. Para que esta información pueda sumarse a los embeddings, deberán tener la misma dimensión *d_model*.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):

        # TO DO: Añadir la codificación posicional al embedding
        sum = ...
        return sum

### 1.5 Modelo completo (versión reducida)

A continuación creamos el modelo completo para la parte de codificación (encoder):

In [ ]:
class MiniTransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers,
                 num_heads, d_ff, max_len):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.encoder = TransformerEncoder(
            num_layers, d_model, num_heads, d_ff
        )
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.encoder(x)
        return self.fc_out(x)

**PREGUNTAS:**

1. Calcula el número total de parámetros del modelo.
2. ¿Qué parte del modelo concentra más parámetros?

A continuación, vamos a comprobar si el modelo puede aprender una tarea simple. Para ello, entrenaremos el modelo para aprender a predecir el valor máximo en toda la secuencia.

Ejemplo:

    Input:  [3, 1, 8, 2, 5, 4]
    Target: [8, 8, 8, 8, 8, 8]


Esto nos permitirá verificar:
- Que la atención funciona.
- Que los gradientes fluyen correctamente.
- Que las conexiones residuales están bien implementadas.

In [ ]:
vocab_size = 20
seq_len = 6
batch_size = 32

model = MiniTransformerEncoder(
    vocab_size=vocab_size,
    d_model=32,
    num_layers=2,
    num_heads=4,
    d_ff=128,
    max_len=seq_len
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for step in range(300):

    # TO DO: Generar una secuencia aleatoria de tokens (batch_size, seq_len)
    # Tensor de enteros entre 0 y vocab_size
    x = ...

    # TO DO: Definir la etiqueta como el valor máximo de la secuencia
    max_vals = ...
    y = ...

    logits = model(x)

    loss = criterion(
        logits.view(-1, vocab_size),
        y.view(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print("Step:", step, "Loss:", loss.item())

In [ ]:
model.eval()

test_seq = torch.tensor([[3, 7, 2, 15, 1, 9]])

with torch.no_grad():
    print("Secuencia de entrada:")
    print(test_seq)

    logits = model(test_seq)
    preds = torch.argmax(logits, dim=-1)
    print("Predicción del modelo:")
    print(preds)

    # Valor máximo real
    real_max = test_seq.max(dim=1, keepdim=True)[0].repeat(1, seq_len)

    print("Máximo real esperado:")
    print(real_max)

### 1.6 Decoder Layer

Para la parte de la decodificación (decoder), tendríamos que usar la parte del enmascaramiento para no usar los valores futuros, así como atención cruzada (cross-attention) en lugar de auto-atención (self-attention). Con ello, podemos crear la clase para la parte de decoder:

In [ ]:
def generate_causal_mask(seq_len):
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, tgt_mask=None):

        # TO DO: Aplicar masked self-attention
        self_attn_out, _ = ...
        x = self.norm1(x + self.dropout(self_attn_out))

        # TO DO: Aplicar cross-attention
        cross_attn_out, _ = ...
        x = self.norm2(x + self.dropout(cross_attn_out))

        # TO DO: Feed-forward
        ff_out = ...
        x = self.norm3(x + self.dropout(ff_out))

        return x

In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, encoder_output, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, encoder_output, tgt_mask)
        return x

### 1.7 Transformer completo

Con todo lo anterior, podemos crear el modelo Transformer completo:

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model,
                 num_layers, num_heads,
                 d_ff, max_len):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        # TO DO: Definir Encoder y Decoder
        self.encoder = ...

        self.decoder = ...

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt):

        src = self.embedding(src)
        src = self.pos_encoding(src)
        encoder_output = self.encoder(src)

        tgt = self.embedding(tgt)
        tgt = self.pos_encoding(tgt)

        tgt_mask = generate_causal_mask(tgt.size(1))

        decoder_output = self.decoder(
            tgt, encoder_output, tgt_mask
        )

        return self.fc_out(decoder_output)

**PREGUNTAS:**
1. ¿Por qué el decoder necesita máscara?
2. ¿Qué diferencia hay entre self-attention y cross-attention?



A continuación, construiremos y entrenaremos un Transformer completo que ordene una secuencia de tokens.

Ejemplo

      src: [8, 2, 5, 1, 7, 3]
      tgt: [1, 2, 3, 5, 7, 8]

In [ ]:
def generate_batch(batch_size, seq_len, vocab_size):
    src = torch.randint(1, vocab_size, (batch_size, seq_len))
    tgt = torch.sort(src, dim=1)[0]   # Ordenar la secuencia
    return src, tgt

In [ ]:
vocab_size = 20
seq_len = 6
batch_size = 32

model = Transformer(
    vocab_size=vocab_size,
    d_model=64,
    num_layers=3,
    num_heads=4,
    d_ff=128,
    max_len=seq_len
)

total_params = sum(p.numel() for p in model.parameters())
print("Total parámetros:", total_params)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for step in range(500):
    src, tgt = generate_batch(batch_size, seq_len, vocab_size)

    tgt_input = tgt[:, :-1]
    tgt_output = tgt[:, 1:]

    logits = model(src, tgt_input)

    loss = criterion(
        logits.reshape(-1, vocab_size),
        tgt_output.reshape(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print("Step:", step, "Loss:", loss.item())

In [ ]:
model.eval()

with torch.no_grad():

    src, tgt = generate_batch(5, seq_len, vocab_size)

    tgt_input = tgt[:, :-1]
    logits = model(src, tgt_input)

    preds = torch.argmax(logits, dim=-1)

    print("Secuencia de entrada:")
    print(src)
    print("Predicción del modelo:")
    print(preds)
    print("Salida esperada:")
    print(tgt[:, 1:])

## PARTE 2: Detección de actividad de voz (VAD)

Como se vio en la práctica anterior, la tarea de **detección de actividad de voz** o _voice activity detection_ (VAD) consiste en encontrar los segmentos donde hay voz y no voz.

En esta parte de la práctica, vamos a diseñar un sistema basado en clustering para dos clases, a partir de representaciones o **embeddings** obtenidos por un modelo pre-entrenado basado en **Transformers**.

El sistema se basará en la siguiente estructura:

```
Audio → [Extracción Embeddings] → [Clustering]
```



En esta práctica nos centraremos en la aplicación de Transformers para el modelado de secuencias de embeddings, utilizando el modelo [WavLM](https://huggingface.co/microsoft/wavlm-large)

De forma general, el procesamiento seguirá las siguientes etapas:

1. Enventanado de la señal
2. Extracción de embeddings (WavLM)
3. Clustering (AHC) - 2 clusters (voz / no voz)

Utilizaremos los ejemplos de la práctica anterior (por ejemplo, **audio_sample.wav**).

In [ ]:
from google.colab import files
uploaded = files.upload()

Podemos representar la señal y ver su forma de onda.

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt

audio_file = 'audio_sample (1).wav'
audio, sr = librosa.load(audio_file, sr=16000)
waveform = torch.tensor(audio)

print("Shape:", waveform.shape)
print("Sample rate:", sr)
print("Duración:", len(audio)/sr, "segundos")

plt.figure(figsize=(12,4))
plt.plot(waveform.numpy())
plt.title(f"{audio_file}")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.show()

Para poder procesar la señal de voz, es habitual dividir el audio en segmentos cortos (ventanas) y analizar cada uno de ellos de forma independiente. Se definen 2 parámetros:
  - **Tamaño de ventana**: indica la duración de cada segmento.
  - **Desplazamiento**: indica cuánto se avanza para generar la siguiente ventana. El stride suele ser menor que el tamaño de ventana para obtener ventanas solapadas.

In [ ]:
# Enventanado temporal
window_sec = 0.1    # Tamaño de ventana (segundos)
stride_sec = 0.1   # Desplazamiento (segundos)

window_size = int(window_sec * sr)
stride = int(stride_sec * sr)

segments = []
timestamps = []

for start in range(0, len(waveform) - window_size, stride):

    # TO DO: definir las ventanas
    end = ...
    window = ...
    segments.append(window)
    timestamps.append(start / sr)   # Tiempo de inicio de cada ventana

print("Número de segmentos:", len(segments))

Como se ha mencionado anteriormente, vamos a utilizar el modelo **WavLM** (basado en Transformers) para la extracción de embeddings. Este modelo ha sido entrenado para aprender representaciones del habla genéricas para distintas tarea.

A continuación, cargamos el modelo pre-entrenado:

In [ ]:
# Cargar modelo pre-entrenado (extractor de embeddings)
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

model_id = "microsoft/wavlm-large"

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_id)
model = WavLMModel.from_pretrained(model_id)
model.eval()

Utilizando el modelo pre-entrenado, extraemos un vector de embeddings para cada segmento de audio. Cada embedding captura características del audio y el contenido de la voz.

Es necesario aplicar un **pooling** para obtener un único vector que represente toda la ventana (en vez de por frame). En este ejemplo utilizaremos el promedio (mean pooling).

Sin embargo, es habitual utilizar un pooling con atención para ponderar los frames según su importancia (requiere entrenamiento).

In [ ]:
def extract_embeddings():

    embeddings = []

    for seg in segments:
        inputs = feature_extractor(
            seg,
            sampling_rate=16000,
            return_tensors="pt"
        )

        with torch.no_grad():
            # TO DO: Inferencia
            outputs = ...

        # WavLM produce un embedding por frame
        hidden = outputs.last_hidden_state

        # TO DO: Mean Pooling para obtener un único vector por ventana
        emb = ...

        embeddings.append(emb.squeeze(0))

    embeddings = torch.stack(embeddings)
    embeddings = F.normalize(embeddings, dim=1)

    return embeddings

Utilizaremos una función de suavizado para evitar "saltos" de clase (voz / no voz) bruscos (por ejemplo, errores de clustering de un solo segmento).

In [ ]:
#Suavizado temporal
# Window: cantidad de vecinos antes y después que se consideran para el suavizado
def smooth_labels(labels, window=2):
    smoothed = []
    for i in range(len(labels)):
        start = max(0, i-window)
        end = min(len(labels), i + window + 1)

        # Seleccionamos la etiqueta más frecuente
        smoothed.append(
            np.bincount(labels[start:end]).argmax()
        )
    return np.array(smoothed)

Aplicamos el pipeline de VAD a partir de los embeddings extraídos:

In [ ]:
from sklearn.cluster import AgglomerativeClustering

def vad_pipeline(embeddings):

    # Agrupamos los embeddings
    clustering = AgglomerativeClustering(
        distance_threshold=None,
        n_clusters=2,
        metric='cosine',
        linkage='average'
    )

    # TO DO: Obtención de las predicciones
    pred_labels = ...

    # TO DO: Suavizado
    pred_labels = ...

    return pred_labels

In [ ]:
# Mean pooling
emb_mean = extract_embeddings()
pred_mean = vad_pipeline(emb_mean)


Visualizaremos el resultado utilizando el código de la práctica anterior, alineando las etiquetas predichas para poder visualizarlas (una por muestra):

In [ ]:
fs=16000
pred_mean_aligned=np.repeat(pred_mean, ... ) # TO DO: alinear
pred_mean_aligned.shape

def plot_signal(signal, fs, ylabel="", title=""):
  dur = len(signal)/fs
  step = 1./fs
  t_axis = np.arange(0., dur, step)

  plt.plot(t_axis, signal)
  plt.xlim([0, dur])
  plt.ylabel(ylabel)
  plt.xlabel('Time (seconds)')
  plt.title(title)
  plt.grid(True)

signal=waveform.numpy()
signal = signal/max(abs(signal))
plot_signal(signal,fs)
plot_signal(pred_mean_aligned*2-1, fs, "Amplitude")
plt.show()

**PREGUNTAS:**
1. ¿Cuál es el rendimiento de este sistema en los ejemplos vistos?
2. ¿Cómo incluiría atención para seleccionar los embeddings en vez de hacer una media? ¿Qué datos se necesitarían?